# Taller 2 — Reformulación de un modelo entero mixto
## Selección de métodos de mejoramiento de arenas sueltas (aplicación conceptual: terreno Las Salinas, Viña del Mar)

Investigación de Operaciones · Doctorado en Ingeniería, consorcio UV–UTA

Este cuaderno reproduce de principio a fin las tablas y figuras del informe (`informe/informe.pdf`).
**Ejecútelo completo** (*Kernel → Restart & Run All*). No requiere preparación: la celda 1 instala lo que falte.
Duración: unos 5 a 10 minutos.

| Celda | Contenido |
|---|---|
| 1 | Instalación automática de dependencias |
| 2 | Importaciones y declaración de entorno |
| 3 | Parámetros (Han, 2015; FHWA, 2017) y diseño experimental |
| 4 | Generador de instancias con semilla (Figura 1) |
| 5 | Modelo: F1 (vínculo agregado) o F2 (desagregado) por parámetro |
| 6 | Medición de los siete indicadores |
| 7 | Experimento completo |
| 8 | Verificación numérica de equivalencia |
| 9 | Tablas y Figura 2 del informe |

In [ ]:
# =============================================================================
# CELDA 1 — Instalación automática de dependencias
# Revisa si cada paquete está instalado; si falta, lo instala en el mismo
# Python que ejecuta este cuaderno. Si ya está, no hace nada.
# Pyomo y HiGHS se instalan en la versión fijada del informe; si su Python no la
# admite, se instala la más reciente compatible y se informa.
# =============================================================================
import importlib, subprocess, sys

PAQUETES = {            # nombre pip : versión fijada (None = versión compatible más reciente)
    "pyomo":      "6.10.1",   # lenguaje de modelado algebraico   } determinan los resultados:
    "highspy":    "1.15.1",   # solver HiGHS (libre, sin licencia) } se fijan
    "numpy":      None,       # bibliotecas de apoyo: su versión real queda declarada en la celda 2
    "pandas":     None,
    "matplotlib": None,
}

def instalado(nombre):
    try:
        importlib.import_module(nombre)
        return True
    except ImportError:
        return False

for nombre, version in PAQUETES.items():
    if instalado(nombre):
        print(f"OK          {nombre}")
        continue
    objetivo = f"{nombre}=={version}" if version else nombre
    print(f"Instalando  {objetivo} ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", objetivo])
    if r.returncode != 0:
        print(f"            versión fijada no disponible para Python {sys.version.split()[0]}; "
              f"se instala la más reciente compatible")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", nombre], check=True)
    importlib.invalidate_caches()
    print(f"OK          {nombre} (instalado)")

print("\nListo: todas las dependencias están disponibles.")


## 2. Importaciones y declaración de entorno
Las versiones y el equipo que se imprimen aquí constituyen la declaración de entorno.

In [ ]:
# =============================================================================
# CELDA 2 — Importaciones y declaración de entorno
# Imprime las versiones reales y el equipo: esto es la "declaración de entorno"
# que exige el enunciado (sección 6) y que se copia al informe.
# =============================================================================
import os, platform, re, tempfile, time, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo
import pyomo.environ as pyo
import highspy
from pyomo.contrib.appsi.solvers import Highs

def procesador():
    """Nombre del procesador (platform.processor() viene vacío en Linux)."""
    if os.path.exists("/proc/cpuinfo"):
        for linea in open("/proc/cpuinfo"):
            if linea.startswith("model name"):
                return linea.split(":", 1)[1].strip()
    return platform.processor() or platform.machine()

ENTORNO = {
    "Python":     platform.python_version(),
    "Pyomo":      pyomo.__version__,
    "HiGHS":      highspy.Highs().version(),
    "NumPy":      np.__version__,
    "pandas":     pd.__version__,
    "Sistema":    f"{platform.system()} {platform.release()}",
    "Procesador": procesador(),
    "Núcleos":    os.cpu_count(),
}
for k, v in ENTORNO.items():
    print(f"{k:11s} {v}")

os.makedirs("resultados", exist_ok=True)   # aquí se guardan tablas y figuras


## 3. Parámetros
Todo supuesto numérico está declarado en esta celda: límites de aplicación de cada método (Han, 2015, tabla 1.5), precios unitarios derivados de FHWA (Schaefer et al., 2017, tabla 1-6), tipos de contratista y diseño experimental.

In [ ]:
# =============================================================================
# CELDA 3 — Parámetros del problema y del diseño experimental
# Todo supuesto numérico del estudio está declarado aquí.
# =============================================================================

# --- Métodos de mejoramiento aplicables a arenas (Han, 2015, tabla 1.5) -----
#   H_max  : profundidad máxima de tratamiento [m]
#   FC_max : contenido máximo de finos [%]  (100 = sin restricción)
#   vibra  : True si induce vibraciones/ruido -> excluido junto a estructuras
#            existentes (Han, 2015, §1.5.2)
#   u      : precio unitario base [USD/m3 de suelo tratado], derivado de FHWA
#            (Schaefer et al., 2017, tabla 1-6; precios nov. 2016)
#   f      : costo fijo base de movilización [USD]; FHWA indica "desde cientos
#            de dólares hasta más de 100 000" según la tecnología (supuesto)
METODOS = {
    "CDP": dict(nombre="Compactación dinámica profunda",  H_max=10, FC_max=15,  vibra=True,  u=3.0,   f=80_000),
    "CIR": dict(nombre="Compactación por impacto rápido", H_max=6,  FC_max=100, vibra=True,  u=2.5,   f=40_000),
    "VCP": dict(nombre="Vibrocompactación",               H_max=15, FC_max=15,  vibra=False, u=4.0,   f=120_000),
    "CGR": dict(nombre="Columnas de grava",               H_max=30, FC_max=100, vibra=False, u=25.0,  f=150_000),
    "JGR": dict(nombre="Jet grouting",                    H_max=30, FC_max=100, vibra=False, u=180.0, f=220_000),
}

# --- Tipos de contratista (Han, 2015, §1.5.2: el contratista local reduce la
#     movilización). Factores sobre el costo fijo y sobre el precio unitario.
CONTRATISTAS = {
    "local":         dict(factor_f=0.6, factor_u=1.10),
    "nacional":      dict(factor_f=1.0, factor_u=1.00),
    "internacional": dict(factor_f=1.8, factor_u=0.90),
}

# --- Sitio (Golder Associates, s. f.: predio de ~15,8 ha) --------------------
AREA_SITIO = 158_000      # m2
FRANJA_COLINDANTE = 40    # m: ancho de la franja junto a estructuras existentes

# --- Diseño experimental -----------------------------------------------------
LADOS = [10, 20, 30, 40, 50]   # grilla L x L -> 100 a 2 500 zonas
SEMILLAS = [11, 22, 33]        # tres sitios independientes por tamaño
REPETICIONES = 3               # resoluciones enteras por caso (tiempo medio)
LIMITE_TIEMPO = 120            # s por resolución
OPCIONES_HIGHS = {"threads": 1, "random_seed": 0, "mip_rel_gap": 1e-6}

pd.DataFrame(METODOS).T


## 4. Generador de instancias
Sitio sintético de 15,8 ha dividido en $L\times L$ zonas. Cada zona recibe espesor a tratar $H$, contenido de finos $FC$ y colindancia; luego se filtran los pares zona–oferta admisibles y se calcula $c_{zj}=a_z H_z u_j$. **Auditoría:** la primera línea debe decir `Instancia L20_s11: 400 zonas, 15 ofertas, 4362 pares admisibles`.

In [ ]:
# =============================================================================
# CELDA 4 — Generador de instancias (semilla declarada)
# Un sitio cuadrado de AREA_SITIO se divide en L x L zonas iguales. Cada zona
# recibe un espesor de arena suelta a tratar H, un contenido de finos FC y una
# marca de colindancia. Luego se filtran los pares (zona, oferta) admisibles y
# se calcula su costo c = área x H x u.
# =============================================================================

def campo_suave(rng, x, y, media, amplitud):
    """Campo espacial suave: suma de dos ondas con fase aleatoria, en [0,1]^2."""
    fx, fy = rng.uniform(0, 2 * np.pi, 2)
    return media + amplitud * (np.sin(3 * x + fx) + np.cos(2.5 * y + fy)) / 2

def generar_instancia(L, semilla):
    rng = np.random.default_rng(semilla)
    lado = np.sqrt(AREA_SITIO)                     # m
    area_zona = AREA_SITIO / L**2                  # m2
    i, j = np.meshgrid(np.arange(L), np.arange(L), indexing="ij")
    x, y = (i.ravel() + 0.5) / L, (j.ravel() + 0.5) / L   # centro de zona, en [0,1]

    # Finos: forma espacial aleatoria, escalada para que el 20 % de las zonas
    # supere 15 % (arenas predominantemente limpias, con sectores limosos)
    g = campo_suave(rng, x, y, 0, 1) + rng.normal(0, 0.15, L * L)
    zonas = pd.DataFrame({
        "x_m": x * lado, "y_m": y * lado,
        "H":  np.clip(campo_suave(rng, x, y, 8, 3.5) + rng.normal(0, 0.5, L * L), 4, 12),
        "FC": np.clip(15 + 6 * (g - np.quantile(g, 0.80)) / g.std(), 2, 30),
    })
    zonas["colindante"] = (zonas.x_m > lado - FRANJA_COLINDANTE) | (zonas.y_m < FRANJA_COLINDANTE)

    # Ofertas: método x contratista, con variación de mercado de +-10 %
    filas = []
    for m, d in METODOS.items():
        for c, k in CONTRATISTAS.items():
            filas.append(dict(metodo=m, contratista=c,
                              f=d["f"] * k["factor_f"] * rng.uniform(0.9, 1.1),
                              u=d["u"] * k["factor_u"] * rng.uniform(0.9, 1.1)))
    ofertas = pd.DataFrame(filas)

    # Pares admisibles A y costos c[z, j]
    costo = {}
    for z, zr in zonas.iterrows():
        for jj, of in ofertas.iterrows():
            d = METODOS[of.metodo]
            if zr.H <= d["H_max"] and zr.FC <= d["FC_max"] and not (d["vibra"] and zr.colindante):
                costo[(z, jj)] = area_zona * zr.H * of.u
    assert all(any(k[0] == z for k in costo) for z in zonas.index), "zona sin oferta admisible"

    return dict(nombre=f"L{L}_s{semilla}", L=L, semilla=semilla,
                zonas=zonas, ofertas=ofertas, costo=costo)

# --- Vista de una instancia de ejemplo ----------------------------------------
ej = generar_instancia(20, 11)
z = ej["zonas"]
print(f"Instancia {ej['nombre']}: {len(z)} zonas, {len(ej['ofertas'])} ofertas, "
      f"{len(ej['costo'])} pares admisibles")
print(f"H entre {z.H.min():.1f} y {z.H.max():.1f} m | zonas con FC > 15 %: {(z.FC > 15).mean():.0%} "
      f"| zonas colindantes: {z.colindante.mean():.0%}")

lado = np.sqrt(AREA_SITIO)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
for a, col, tit, cmap in [(ax[0], "H", "Espesor de arena suelta a tratar, H [m]", "viridis"),
                          (ax[1], "FC", "Contenido de finos, FC [%]", "magma_r"),
                          (ax[2], "colindante", "Franja colindante (CDP y CIR excluidas)", "Greys")]:
    im = a.imshow(z[col].astype(float).values.reshape(ej["L"], ej["L"]).T, origin="lower",
                  cmap=cmap, extent=[0, lado, 0, lado])
    a.set_title(tit, fontsize=9.5); a.set_xlabel("x [m]")
    cb = fig.colorbar(im, ax=a, shrink=0.8)
    if col == "colindante":
        cb.ax.set_visible(False)      # dato sí/no: sin barra, pero mismo tamaño de panel
    if col == "FC":   # contorno del umbral de 15 % de finos (Han, 2015)
        a.contour(z.FC.values.reshape(ej["L"], ej["L"]).T, levels=[15], colors="white",
                  linewidths=1.2, extent=[0, lado, 0, lado], origin="lower")
for a in ax:
    a.set_xlim(0, 400); a.set_ylim(0, 400)
    a.set_xticks(range(0, 401, 100)); a.set_yticks(range(0, 401, 100))
ax[0].set_ylabel("y [m]")
from matplotlib.patches import Patch
ax[2].legend(handles=[Patch(facecolor="black", label="Estructuras existentes (condición de borde)"),
                      Patch(facecolor="white", edgecolor="black", label="Perímetro libre")],
             loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=8, frameon=False)
fig.suptitle(f"Área total del sitio: {AREA_SITIO/1e4:.1f}".replace(".", ",") + f" ha (≈ {lado:.0f} m × {lado:.0f} m), "
             f"{ej['L']} × {ej['L']} = {ej['L']**2} zonas de ≈ {lado/ej['L']:.0f} m × {lado/ej['L']:.0f} m",
             fontsize=10.5)
plt.tight_layout()
plt.savefig("resultados/figura_sitio.png", dpi=200, bbox_inches="tight")
plt.show()

ej["ofertas"].round(2)


## 5. Modelo
$$\min \sum_j f_j y_j + \sum_{(z,j)\in A} c_{zj}x_{zj}\quad \text{s.a.}\quad \sum_{j\in J_z} x_{zj}=1\ \forall z$$
- **F1 (agregada):** $\sum_{z\in Z_j} x_{zj} \le |Z_j|\,y_j\quad \forall j$
- **F2 (desagregada):** $x_{zj}\le y_j\quad \forall (z,j)\in A$

La celda termina con una prueba: si F1 y F2 dieran óptimos distintos, se detiene con error.

In [ ]:
# =============================================================================
# CELDA 5 — Modelo MILP: F1 (vínculo agregado) o F2 (vínculo desagregado)
# Ambas formulaciones comparten datos, variables, objetivo (1) y asignación (2);
# solo cambia la restricción de vínculo, que se elige con el parámetro
# `formulacion`. La numeración sigue el informe (sección 2).
# =============================================================================

def construir_modelo(inst, formulacion):
    assert formulacion in ("F1", "F2")
    costo = inst["costo"]                         # {(z, j): c_zj} solo pares admisibles A
    f = inst["ofertas"]["f"].to_dict()            # {j: f_j}
    Z, J = list(inst["zonas"].index), list(inst["ofertas"].index)
    J_z = {z: [] for z in Z}                      # ofertas admisibles en la zona z
    Z_j = {j: [] for j in J}                      # zonas donde la oferta j es admisible
    for (z, j) in costo:
        J_z[z].append(j)
        Z_j[j].append(z)

    m = pyo.ConcreteModel(name=f"{inst['nombre']}_{formulacion}")
    m.A = pyo.Set(initialize=list(costo), dimen=2)
    m.J = pyo.Set(initialize=J)
    m.x = pyo.Var(m.A, domain=pyo.Binary)         # x_zj: zona z tratada con la oferta j
    m.y = pyo.Var(m.J, domain=pyo.Binary)         # y_j : oferta j movilizada

    # (1) movilización + tratamiento
    m.obj = pyo.Objective(sense=pyo.minimize,
        expr=sum(f[j] * m.y[j] for j in J) + sum(c * m.x[z, j] for (z, j), c in costo.items()))
    # (2) cada zona se trata con exactamente una oferta admisible
    m.asignacion = pyo.Constraint(Z, rule=lambda m, z: sum(m.x[z, j] for j in J_z[z]) == 1)

    if formulacion == "F1":   # (4a) una restricción por oferta
        m.vinculo = pyo.Constraint([j for j in J if Z_j[j]],
            rule=lambda m, j: sum(m.x[z, j] for z in Z_j[j]) <= len(Z_j[j]) * m.y[j])
    else:                     # (4b) una restricción por par admisible
        m.vinculo = pyo.Constraint(m.A, rule=lambda m, z, j: m.x[z, j] <= m.y[j])
    return m


def resolver(m, relajar=False, limite=LIMITE_TIEMPO, log=None):
    """Resuelve con HiGHS bajo OPCIONES_HIGHS. relajar=True suprime la integralidad
    (cota de la relajación lineal). Devuelve el resultado y el objeto HiGHS interno,
    de donde se leen nodos, cota dual, brecha y dimensiones."""
    opt = Highs()
    opt.config.time_limit = limite
    opt.config.stream_solver = False
    opt.config.load_solution = False
    opciones = dict(OPCIONES_HIGHS)
    if relajar:
        opciones["solve_relaxation"] = True
    if log:
        opciones.update(log_file=log, log_to_console=False)
    opt.highs_options = opciones
    res = opt.solve(m)
    if not relajar and res.best_feasible_objective is not None:
        res.solution_loader.load_vars()           # deja la solución en m.x, m.y
    return res, opt._solver_model


# --- Prueba en una instancia chica: ¿dan F1 y F2 el mismo óptimo? -------------
inst = generar_instancia(10, 11)
prueba, exacto = {}, {}
for F in ("F1", "F2"):
    m = construir_modelo(inst, F)
    _, h_lp = resolver(m, relajar=True)
    _, h_ip = resolver(m)
    exacto[F] = (h_lp.getInfo().objective_function_value, h_ip.getInfo().objective_function_value)
    abiertas = [j for j in m.J if pyo.value(m.y[j]) > 0.5]
    prueba[F] = {"restricciones": h_ip.getNumRow(), "variables": h_ip.getNumCol(),
                 "z_LP [MUSD]": round(h_lp.getInfo().objective_function_value / 1e6, 4),
                 "z_IP [MUSD]": round(h_ip.getInfo().objective_function_value / 1e6, 4),
                 "nodos": h_ip.getInfo().mip_node_count, "tiempo [s]": round(h_ip.getRunTime(), 3)}
    print(f"{F} moviliza: " + ", ".join(
        f"{inst['ofertas'].metodo[j]}-{inst['ofertas'].contratista[j]}" for j in abiertas))
display(pd.DataFrame(prueba).T)

dif = abs(exacto["F1"][1] - exacto["F2"][1]) / exacto["F1"][1]
assert dif < 1e-6, "F1 y F2 dan óptimos distintos: revisar el modelo"
assert exacto["F2"][0] >= exacto["F1"][0] - 1e-6, "Viola la Proposición 1"
print(f"OK: óptimo idéntico (diferencia relativa {dif:.1e}) y z_LP(F2) >= z_LP(F1).")


## 6. Medición de los siete indicadores
Cota de la relajación, óptimo entero, brecha de la relajación, nodos, cortes (del registro de HiGHS), tiempo (media y desviación de 3 corridas) y dimensión del modelo. La cota al terminar el nodo raíz se obtiene deteniendo HiGHS tras un nodo.

In [ ]:
# =============================================================================
# CELDA 6 — Medición de los siete indicadores para un caso (instancia x formulación)
#   1 cota de la relajación lineal   2 óptimo entero   3 brecha de la relajación
#   4 nodos   5 cortes (generados / activos)   6 tiempo (media y desv. de R corridas)
#   7 dimensión (variables, restricciones, no nulos)
# Los cortes se leen del registro (log) de HiGHS: columnas "Cuts" (generados, en el
# pool) e "InLp" (activos en el LP). La cota al cerrar el nodo raíz se obtiene
# deteniendo HiGHS tras un nodo.
# =============================================================================
import uuid

# Fila de la tabla de ramificación del registro de HiGHS:
# Src Proc. InQueue | Leaves Expl. | BestBound BestSol Gap | Cuts InLp Confl. | LpIters Time
FILA_BB = re.compile(r"^\s*[A-Za-z]?\s+\d+\s+\d+\s+\d+\s+[\d.]+%\s+\S+\s+\S+\s+\S+\s+"
                     r"(\d+)\s+(\d+)\s+\d+\s+\d+\s+[\d.]+s")

def leer_cortes(ruta):
    """(cortes generados, cortes activos en el LP) de la última fila de la tabla."""
    filas = [f for f in (FILA_BB.match(l) for l in open(ruta, encoding="utf-8", errors="replace")) if f]
    return (int(filas[-1].group(1)), int(filas[-1].group(2))) if filas else (0, 0)

def cota_nodo_raiz(m, limite):
    """Cota dual al terminar el nodo raíz (relajación + cortes automáticos), obtenida
    deteniendo HiGHS tras procesar un solo nodo. Mide lo que cierra la separación."""
    opt = Highs()
    opt.config.time_limit = limite
    opt.config.stream_solver = False
    opt.config.load_solution = False
    opt.highs_options = dict(OPCIONES_HIGHS, mip_max_nodes=1)
    opt.solve(m)
    return opt._solver_model.getInfo().mip_dual_bound


def medir(inst, formulacion, repeticiones=REPETICIONES, limite=LIMITE_TIEMPO):
    m = construir_modelo(inst, formulacion)
    _, h_lp = resolver(m, relajar=True, limite=limite)              # indicador 1
    z_lp = h_lp.getInfo().objective_function_value

    tiempos = []
    for r in range(repeticiones):                                  # indicador 6
        # archivo de registro nuevo en cada corrida (en Windows no se puede borrar
        # un archivo que otro proceso mantiene abierto)
        log = os.path.join(tempfile.gettempdir(), f"highs_{uuid.uuid4().hex}.log")
        res, h = resolver(m, limite=limite, log=log)
        h.setOptionValue("log_file", "")                           # HiGHS cierra y suelta el registro
        tiempos.append(h.getRunTime())
        if r < repeticiones - 1:
            os.remove(log)
    info = h.getInfo()                                             # última repetición
    cortes, activos = leer_cortes(log)                             # indicador 5
    os.remove(log)
    cota_raiz = cota_nodo_raiz(m, limite)
    z_ip = info.objective_function_value                           # indicador 2 (o mejor solución)

    return {
        "instancia": inst["nombre"], "zonas": len(inst["zonas"]), "semilla": inst["semilla"],
        "formulacion": formulacion,
        "variables": h.getNumCol(), "restricciones": h.getNumRow(), "no_nulos": h.getNumNz(),  # 7
        "z_LP": z_lp, "z_IP": z_ip,
        "brecha_LP_%": 100 * (z_ip - z_lp) / z_ip,                 # indicador 3
        "integrality_gap": z_ip / z_lp,
        "nodos": info.mip_node_count,                              # indicador 4
        "cortes": cortes, "cortes_activos": activos,
        "cerrada_raiz_%": (100 * (min(cota_raiz, z_ip) - z_lp) / (z_ip - z_lp)
                           if z_ip - z_lp > 1e-6 * abs(z_ip) else np.nan),   # sin brecha: no aplica
        "t_medio_s": statistics.mean(tiempos), "t_desv_s": statistics.stdev(tiempos) if len(tiempos) > 1 else 0.0,
        "estado": res.termination_condition.name,
        "brecha_opt_%": 100 * info.mip_gap,                        # solo > 0 si se alcanzó el límite
        "ofertas_movilizadas": sum(round(pyo.value(m.y[j])) for j in m.J),
    }

# --- Prueba sobre la instancia de la celda 5 ----------------------------------
prueba = pd.DataFrame([medir(inst, F) for F in ("F1", "F2")]).set_index("formulacion")
display(prueba.T)


## 7. Experimento completo
5 tamaños × 3 sitios × 2 formulaciones × 3 repeticiones, bajo condiciones idénticas. Resultado en `resultados/resultados.csv`.

In [ ]:
# =============================================================================
# CELDA 7 — Experimento completo
# 5 tamaños x 3 sitios x 2 formulaciones, cada una con REPETICIONES resoluciones
# enteras bajo condiciones idénticas. Se guarda en resultados/resultados.csv.
# Duración aproximada: 5 a 10 minutos (depende del equipo).
# =============================================================================
filas = []
t0 = time.time()
for L in LADOS:
    for s in SEMILLAS:
        inst = generar_instancia(L, s)
        for F in ("F1", "F2"):
            r = medir(inst, F)
            filas.append(r)
            print(f"[{time.time() - t0:6.0f} s] {inst['nombre']:8s} {F}  {r['estado']:8s} "
                  f"z_IP = {r['z_IP'] / 1e6:8.4f} MUSD  brecha LP = {max(r['brecha_LP_%'], 0):5.2f} %  "
                  f"nodos = {r['nodos']:3d}  t = {r['t_medio_s']:6.2f} s")

res = pd.DataFrame(filas)
res.to_csv("resultados/resultados.csv", index=False)
print(f"\nListo: {len(res)} casos en {(time.time() - t0) / 60:.1f} min -> resultados/resultados.csv")


## 8. Verificación numérica de equivalencia
Complementa la demostración algebraica del informe (sección 3).

In [ ]:
# =============================================================================
# CELDA 8 — Verificación numérica de equivalencia en todas las instancias
# Complementa el argumento algebraico (informe, sección 3):
#   (a) mismo óptimo entero en F1 y F2 (requisito previo a toda comparación)
#   (b) z_LP(F2) >= z_LP(F1)  (Proposición 1)
# =============================================================================
f1 = res[res.formulacion == "F1"].set_index("instancia")
f2 = res[res.formulacion == "F2"].set_index("instancia")
eq = pd.DataFrame({
    "z_IP F1 [MUSD]": f1.z_IP / 1e6,
    "z_IP F2 [MUSD]": f2.z_IP / 1e6,
    "dif. relativa": (f1.z_IP - f2.z_IP).abs() / f1.z_IP,
    "ambas óptimas": (f1.estado == "optimal") & (f2.estado == "optimal"),
    "z_LP F2 >= z_LP F1": f2.z_LP >= f1.z_LP - 1e-6 * f1.z_LP.abs(),
})
eq.to_csv("resultados/equivalencia.csv")
display(eq.round(6))

assert eq["ambas óptimas"].all(), "Hay instancias sin optimalidad probada: revisar el límite de tiempo"
assert (eq["dif. relativa"] < 1e-6).all(), "Óptimos distintos: la comparación no es válida"
assert eq["z_LP F2 >= z_LP F1"].all(), "Se viola la Proposición 1"
print(f"OK: {len(eq)} instancias con óptimo idéntico (diferencia relativa máx. "
      f"{eq['dif. relativa'].max():.1e}) y z_LP(F2) >= z_LP(F1) en todas.")


## 9. Tablas y figura del informe

In [ ]:
# =============================================================================
# CELDA 9 — Tablas y figura del informe (sección 6)
# Tabla 1: siete indicadores por instancia y formulación.
# Tabla 2: resumen por tamaño (promedio de los tres sitios).
# Figura 2: brecha de la relajación y tiempo de resolución frente al tamaño.
# =============================================================================

# --- Tabla 1 -----------------------------------------------------------------
t1 = pd.DataFrame({
    "Zonas": res.zonas, "Sem.": res.semilla, "F": res.formulacion,
    "Var.": res.variables, "Restr.": res.restricciones, "No nulos": res.no_nulos,
    "z_LP [MUSD]": (res.z_LP / 1e6).round(4), "z_IP [MUSD]": (res.z_IP / 1e6).round(4),
    "Brecha LP [%]": res["brecha_LP_%"].clip(lower=0).round(2),   # recorta ruido numérico (-1e-12)
    "Nodos": res.nodos, "Cortes gen./act.": res.cortes.astype(str) + "/" + res.cortes_activos.astype(str),
    "Cerrada raíz [%]": res["cerrada_raiz_%"].round(1),
    "t medio [s]": res.t_medio_s.round(3), "t desv. [s]": res.t_desv_s.round(3),
    "Estado": res.estado,
})
t1.to_csv("resultados/tabla1_indicadores.csv", index=False)
display(t1)

# --- Tabla 2 -----------------------------------------------------------------
par = f1[["zonas"]].copy()
par["brecha_F1"] = f1["brecha_LP_%"]
par["brecha_F2"] = f2["brecha_LP_%"].clip(lower=0)
par["cerrada_raiz_F1"] = f1["cerrada_raiz_%"]
par["nodos_F1"], par["nodos_F2"] = f1.nodos, f2.nodos
par["cortes_F1"], par["cortes_F2"] = f1.cortes, f2.cortes
par["t_F1"], par["t_F2"] = f1.t_medio_s, f2.t_medio_s
par["aceleracion"] = f1.t_medio_s / f2.t_medio_s          # >1: F2 más rápida
par["restr_F2/F1"] = f2.restricciones / f1.restricciones

t2 = par.groupby("zonas").agg(**{
    "Brecha LP F1 [%]": ("brecha_F1", "mean"), "Brecha LP F2 [%]": ("brecha_F2", "mean"),
    "Cerrada raíz F1 [%]": ("cerrada_raiz_F1", "mean"),
    "Nodos F1": ("nodos_F1", "mean"), "Nodos F2": ("nodos_F2", "mean"),
    "Cortes F1": ("cortes_F1", "mean"), "Cortes F2": ("cortes_F2", "mean"),
    "t F1 [s]": ("t_F1", "mean"), "t F2 [s]": ("t_F2", "mean"),
    "t F1/t F2 (geom.)": ("aceleracion", lambda s: float(np.exp(np.log(s).mean()))),
    "Restr. F2/F1": ("restr_F2/F1", "mean"),
}).round(2)
t2.to_csv("resultados/tabla2_resumen.csv")
display(t2)

# --- Figura 2 ----------------------------------------------------------------
COLOR = {"F1": "#2a78d6", "F2": "#eb6834"}
MARCA = {"F1": "o", "F2": "s"}
ETIQ = {"F1": "F1 (vínculo agregado)", "F2": "F2 (vínculo desagregado)"}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for F, d in (("F1", f1), ("F2", f2)):
    g = d.groupby("zonas")
    brecha = d["brecha_LP_%"].clip(lower=0)
    ax[0].scatter(d.zonas, brecha, color=COLOR[F], marker=MARCA[F], s=22, alpha=0.4)
    ax[0].plot(g.size().index, brecha.groupby(d.zonas).mean(), color=COLOR[F], marker=MARCA[F],
               lw=2, ms=7, label=ETIQ[F])
    ax[1].scatter(d.zonas, d.t_medio_s, color=COLOR[F], marker=MARCA[F], s=22, alpha=0.4)
    ax[1].plot(g.size().index, g.t_medio_s.mean(), color=COLOR[F], marker=MARCA[F],
               lw=2, ms=7, label=ETIQ[F])
ax[0].set(xlabel="Número de zonas", ylabel="Brecha de la relajación [%]",
          title="(a) Brecha de la relajación lineal")
ax[1].set(xlabel="Número de zonas", ylabel="Tiempo de resolución [s] (escala log)",
          title="(b) Tiempo de resolución (media de 3 corridas)", yscale="log")
for a in ax:
    a.grid(alpha=0.25, lw=0.6)
    a.spines[["top", "right"]].set_visible(False)
    a.legend(frameon=False, fontsize=9)
    a.set_xticks(sorted(res.zonas.unique()))
plt.tight_layout()
plt.savefig("resultados/figura_resultados.png", dpi=200, bbox_inches="tight")
plt.show()
print("Guardado en resultados/: tabla1_indicadores.csv, tabla2_resumen.csv, figura_resultados.png")
